# Webcam Recorder Widget

Records webcam **video + audio** to disk, with live filters baked in.

Run the next cell, then click **Start Camera** → **Record** → **Stop**. The clip is saved automatically.

In [1]:
from webcam_recorder import WebcamRecorderWidget

w = WebcamRecorderWidget(save_dir="recordings", filename_prefix="take")
w

In [2]:
# Where did the last clip land?
w.last_saved_path

''

## Drive it from Python (timed capture)

In [29]:
import time

w.record()          # start camera (if needed) + recording


In [30]:
w.stop_recording()  # flush to disk
print(w.last_saved_path)

In [31]:
print(w.last_saved_path)

## Post-processing (optional, requires ffmpeg)

In [32]:
w.to_mp4()          # -> H.264/AAC .mp4 next to the .webm
# w.extract_audio() # -> standalone .m4a

FileNotFoundError: No recording to convert yet — record a clip first (w.record() / w.stop_recording()).

## Send the recording into Blender's Video Sequence Editor

Run this **inside Blender's Python**. It saves the last clip as an mp4 in your Downloads folder, then drops it into the next free channel of the VSE.

In [22]:
import pathlib


def add_recording_to_vse(widget, directory=None, *, with_audio=True, frame_start=None):
    """Save the widget's latest recording as mp4, then add it to Blender's VSE.

    1. Transcodes the last clip to H.264/AAC mp4 in `directory`
       (default: your ~/Downloads folder).
    2. Adds it as a movie strip in the next free channel of the Video Sequence
       Editor (its audio goes on the channel above when `with_audio`).

    `frame_start` defaults to the scene's current frame. Returns the movie strip.
    Run this inside Blender's bundled Python, where `bpy` is available.
    """
    try:
        import bpy
    except ImportError as exc:
        print("nope")
        raise RuntimeError(
            "bpy not available — run this notebook inside Blender's Python."
        ) from exc

    target_dir = (
        pathlib.Path(directory).expanduser()
        if directory
        else pathlib.Path.home() / "Downloads"
    )
    mp4_path = pathlib.Path(widget.to_mp4(dest=target_dir))

    scene = bpy.context.scene
    if not scene.sequence_editor:
        scene.sequence_editor_create()
    sed = scene.sequence_editor

    # Blender 4.4 renamed `sequences*` -> `strips*`; support both.
    all_strips = getattr(sed, "strips_all", None)
    if all_strips is None:
        all_strips = getattr(sed, "sequences_all", [])
    channel = max((s.channel for s in all_strips), default=0) + 1
    start = scene.frame_current if frame_start is None else int(frame_start)

    container = getattr(sed, "strips", None) or sed.sequences
    movie = container.new_movie(
        name=mp4_path.stem, filepath=str(mp4_path), channel=channel, frame_start=start
    )
    if with_audio:
        container.new_sound(
            name=f"{mp4_path.stem}_audio",
            filepath=str(mp4_path),
            channel=channel + 1,
            frame_start=start,
        )
    print(f"Added '{mp4_path.name}' to VSE channel {channel} at frame {start}.")
    return movie


# Record a clip first (Start Camera -> Record -> Stop, or w.record()/w.stop_recording()),
# then drop it straight into the sequencer:
add_recording_to_vse(w)

nope


RuntimeError: bpy not available — run this notebook inside Blender's Python.